In [1]:
# Imports
import os
from pathlib import Path
markers = (".git", "Program")
current_dir = Path.cwd()
project_root = next((path for path in (current_dir, *current_dir.parents) if any((path / marker).exists() for marker in markers)), current_dir)
os.chdir(project_root)

import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_infix
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = False
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix("^GSPC", index_dict, all_stocks) 

# Modify the current date
current_date = modify_current_date(start, index_name)

In [2]:
# Index
indices = ["^HSI", "^GSPC", "^IXIC", "QLD", "TQQQ"]
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite", "QLD": "ProShares Ultra QQQ", "TQQQ": "ProShares UltraPro QQQ"}

# Momentum ETFs
etfs_dict = {
    "MTUM": "iShares MSCI USA Momentum Factor ETF",
    "IMTM": "iShares MSCI Intl Momentum Factor ETF",
    "SMLF": "iShares U.S. Small‑Cap Equity Factor ETF",
    "IUSV": "iShares S&P U.S. Value ETF",
    "IJJ": "iShares S&P Mid-Cap 400 Value ETF",
    "IJS": "iShares S&P Small-Cap 600 Value ETF",
    "EFV": "iShares MSCI EAFE Value ETF",
    "SPMO": "Invesco S&P 500 Momentum ETF",
    "XMMO": "Invesco S&P MidCap Momentum ETF",
    "XSMO": "Invesco S&P SmallCap Momentum ETF",
    "PDP": "Invesco Dorsey Wright Momentum ETF",
    "IDMO": "Invesco S&P Intl Developed Momentum ETF",
    "RPV": "Invesco S&P 500 Pure Value ETF",
    "SPHD": "Invesco S&P 500 High Dividend Low Volatility ETF",
    "VFMO": "Vanguard U.S. Momentum Factor ETF",
    "VYM": "Vanguard High Dividend Yield ETF",
    "VOE": "Vanguard S&P Mid-Cap Value ETF",
    "VONV": "Vanguard Russell 1000 Value ETF",
    "VOOV": "Vanguard S&P 500 Value ETF",
    "MGV": "Vanguard Mega Cap Value ETF",
    "VBR": "Vanguard Small-Cap Value ETF",
    "VTV": "Vanguard Value ETF",
    "AVLV": "Avantis U.S. Large Cap Value ETF",
    "AVUV": "Avantis U.S. Small Cap Value ETF",
    "JMOM": "JPMorgan U.S. Momentum Factor ETF",
    "GLOV": "Activebeta World Low Vol Plus Equity ETF",
    "DWMF": "WisdomTree International Multifactor ETF",
    "AFLG": "First Trust Active Factor Large Cap Intl ETF",
    "CGDV": "Capital Group Dividend Value ETF",
    "SCHD": "Schwab U.S. Dividend Equity ETF",
    "3070.HK": "Ping An of China CSI HK Dividend ETF"
}
etfs = list(etfs_dict.keys())

In [3]:
# Create an empty DataFrame to store the data of the ETFs
etfs_data = pd.DataFrame(columns=[
    "Symbol", "Establishment Date", "1 Year CAGR (%)", "3 Year CAGR (%)", 
    "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)", "Max Period CAGR (%)", 
    "Max Period Annual Volatility (%)", "Max Period Sharpe", 
    "Max Period Sortino", "Max Period MDD (%)", "Max Period Calmar", 
    "5 Year Annual Volatility (%)", "5 Year Sharpe", "5 Year Sortino", 
    "5 Year MDD (%)", "5 Year Calmar", "5 Year Peak Date", 
    "5 Year Recovery Date", "5 Year Recovery Period (days)"
])

# Loop through each ETF in the indices and momentum ETFs
for etf in tqdm(indices + etfs):
    # Get the name of the ETF
    etf_name = {**index_dict, **etfs_dict}.get(etf, etf)
    etfs_data.loc[etf_name, "Symbol"] = etf

    # Get the dataframe for the ETF
    df = get_df(etf, current_date, max_period=True, adj=True)
    
    # Get the establishment date of the ETF
    establishment_date = df.index[0].date()
    etfs_data.loc[etf_name, "Establishment Date"] = establishment_date
    
    # Calculate CAGR values for different periods
    for period, label in zip([252, 252 * 3, 252 * 5, 252 * 10, 252 * 20], 
                             ["1 Year CAGR (%)", "3 Year CAGR (%)", "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)"]):
        if len(df) >= period:
            start_price = df["Close"].iloc[- period]
            end_price = df["Close"].iloc[- 1]
            cagr = ((end_price / start_price) ** (1 / (period / 252)) - 1) * 100
            etfs_data.loc[etf_name, label] = cagr

    # Calculate CAGR of maximum period
    start_price = df["Close"].iloc[0]
    end_price = df["Close"].iloc[- 1]
    max_period_cagr = ((end_price / start_price) ** (1 / (len(df) / 252)) - 1) * 100
    etfs_data.loc[etf_name, "Max Period CAGR (%)"] = max_period_cagr

    # Calculate percent change and cumulative return
    df["Percent Change"] = df["Close"].pct_change().dropna()
    df["Cumulative Return"] = (1 + df["Percent Change"]).cumprod()
    daily_returns = df["Percent Change"]
    cumulative_returns = df["Cumulative Return"]
        
    # Calculate annual volatility
    annual_volatility = daily_returns.std() * np.sqrt(252) * 100
    etfs_data.loc[etf_name, "Max Period Annual Volatility (%)"] = annual_volatility

    # Calculate 5 year annual volatility
    if len(df) >= 252 * 5:
        daily_returns_5y = daily_returns.iloc[- 252 * 5:]
        annual_volatility_5y = daily_returns_5y.std() * np.sqrt(252) * 100
        etfs_data.loc[etf_name, "5 Year Annual Volatility (%)"] = annual_volatility_5y

    # Calculate Sharpe ratio
    risk_free_rate = 0.03  # Assuming a risk-free rate of 3%
    sharpe_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (daily_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sharpe"] = sharpe_ratio

    # Calculate 5 year Sharpe ratio
    if len(df) >= 252 * 5:
        sharpe_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (daily_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sharpe"] = sharpe_ratio_5y

    # Calculate Sortino ratio
    negative_returns = daily_returns[daily_returns < 0]
    sortino_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (negative_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sortino"] = sortino_ratio

    # Calculate 5 year Sortino ratio
    if len(df) >= 252 * 5:
        negative_returns_5y = daily_returns_5y[daily_returns_5y < 0]
        sortino_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (negative_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sortino"] = sortino_ratio_5y

    # Calculate maximum drawdown
    df["Peak"] = df["Cumulative Return"].cummax()
    df["Drawdown"] = (df["Cumulative Return"] - df["Peak"]) / df["Peak"]
    drawdowns = df["Drawdown"]
    mdd = drawdowns.min() * 100
    mdd_date = drawdowns.idxmin()
    etfs_data.loc[etf_name, "Max Period MDD (%)"] = mdd

    # Calculate 5 year maximum drawdown
    if len(df) >= 252 * 5:
        drawdowns_5y = drawdowns[- 252 * 5:]
        mdd_5y = drawdowns_5y.min() * 100
        etfs_data.loc[etf_name, "5 Year MDD (%)"] = mdd_5y
        mdd_date_5y = drawdowns_5y.idxmin()

        # Find the peak date before the maximum drawdown
        peak_mask_5y = (df.index <= mdd_date_5y) & (df["Cumulative Return"] == df["Peak"])
        peak_date_5y = df.index[peak_mask_5y][-1]
        etfs_data.loc[etf_name, "5 Year Peak Date"] = peak_date_5y.date()

        # Find the recovery date after the maximum drawdown
        rec_mask_5y = (df.index > mdd_date_5y) & (df["Cumulative Return"] >= df.loc[peak_date_5y, "Cumulative Return"])
        rec_date_5y = df.index[rec_mask_5y][0] if len(df.index[rec_mask_5y]) > 0 else None
        if rec_date_5y is not None:
            etfs_data.loc[etf_name, "5 Year Recovery Date"] = rec_date_5y.date()
            rec_period_5y = (rec_date_5y - peak_date_5y).days
            etfs_data.loc[etf_name, "5 Year Recovery Period (days)"] = rec_period_5y

        # Calculate the two most significant maximum drawdown in the 5 year period
        drawdowns_5y = drawdowns_5y.sort_values()
        mdd_5y = drawdowns_5y.iloc[0] * 100

    # Calculate Calmar ratio
    calmar_ratio = (daily_returns.mean() * 252 - risk_free_rate) / abs(mdd / 100)
    etfs_data.loc[etf_name, "Max Period Calmar"] = calmar_ratio

    # Calculate 5 year Calmar ratio
    if len(df) >= 252 * 5:
        calmar_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / abs(mdd_5y / 100)
        etfs_data.loc[etf_name, "5 Year Calmar"] = calmar_ratio_5y

100%|██████████| 36/36 [00:24<00:00,  1.45it/s]


In [4]:
etfs_data

,Symbol,Establishment Date,1 Year CAGR (%),3 Year CAGR (%),5 Year CAGR (%),10 Year CAGR (%),20 Year CAGR (%),Max Period CAGR (%),Max Period Annual Volatility (%),Max Period Sharpe,...,Max Period MDD (%),Max Period Calmar,5 Year Annual Volatility (%),5 Year Sharpe,5 Year Sortino,5 Year MDD (%),5 Year Calmar,5 Year Peak Date,5 Year Recovery Date,5 Year Recovery Period (days)
Hang Seng Index,^HSI,1986-12-31,39.016904,10.779032,0.690923,1.738556,2.903815,6.377781,25.596939,0.25446,...,-65.18186,0.099927,25.110868,0.041387,0.061827,-55.700772,0.018658,2018-01-26,NaN,NaN
S&P 500,^GSPC,1927-12-30,14.294771,19.557156,12.506127,13.967651,8.797547,6.300652,18.934222,0.259169,...,-86.189579,0.056934,16.924948,0.601861,0.822888,-25.425097,0.400646,2022-01-03,2024-01-19,746
NASDAQ Composite,^IXIC,1971-02-05,19.205967,26.784205,11.48034,17.906252,12.309013,10.430276,20.133688,0.444712,...,-77.932386,0.11489,22.603809,0.460286,0.644835,-36.39528,0.285867,2021-11-19,2024-02-29,832
ProShares Ultra QQQ,QLD,2006-06-21,27.839474,50.848865,18.169664,33.448947,NaN,24.491437,43.951335,0.651205,...,-83.128875,0.344301,45.164061,0.529355,0.744001,-63.684472,0.375411,2021-11-19,2024-05-24,917
ProShares UltraPro QQQ,TQQQ,2010-02-11,30.931721,70.445034,16.899847,41.316844,NaN,41.797977,61.014564,0.832075,...,-81.659848,0.62171,67.130009,0.523746,0.734504,-81.659848,0.430555,2021-11-19,2024-12-04,1111
iShares MSCI USA Momentum Factor ETF,MTUM,2013-04-18,17.507331,21.800273,9.835392,15.443566,NaN,14.852777,19.480467,0.654939,...,-34.082302,0.374345,20.633718,0.401996,0.555905,-32.284553,0.256924,2021-11-03,2024-03-04,852
iShares MSCI Intl Momentum Factor ETF,IMTM,2015-01-27,34.604972,19.794662,9.28765,11.01699,NaN,9.038039,19.930591,0.382456,...,-30.683177,0.248429,17.018913,0.424998,0.624221,-30.683177,0.235732,2021-11-08,2024-02-22,836
iShares U.S. Small‑Cap Equity Factor ETF,SMLF,2015-04-30,10.580356,14.510746,10.844758,12.70926,NaN,10.891767,21.653412,0.448035,...,-41.892206,0.231582,21.310333,0.442579,0.687015,-26.277109,0.358925,2024-11-25,2025-08-28,276
iShares S&P U.S. Value ETF,IUSV,2000-08-04,11.833286,14.098286,13.016911,13.08824,8.705299,8.383645,19.091914,0.360365,...,-60.183515,0.114318,14.735132,0.696348,0.998137,-17.951535,0.571582,2022-04-20,2023-02-01,287
iShares S&P Mid-Cap 400 Value ETF,IJJ,2000-07-28,6.408152,9.542769,10.666821,12.164369,8.810159,10.253056,21.855766,0.419168,...,-58.003744,0.157942,19.82987,0.451358,0.693833,-22.679144,0.394652,2024-11-25,2025-12-10,380


In [5]:
# Save the data as a CSV file
result_folder = "Result"
filename = os.path.join(result_folder, "etfs_comparison.csv")
etfs_data.to_csv(filename)